In [0]:
from pyspark.sql import functions as F

In [0]:
schema = 'finance'
table_name = 'dim_taxonomy'

# dbutils.widgets.text("year", "", "GAAP Version Year (leave blank for all versions)")
# gaap_year_to_process = dbutils.widgets.get("year")

dbutils.widgets.text("target_catalog", "", "Target Catalog")
target_catalog = dbutils.widgets.get("target_catalog")

In [0]:
df = spark.table("operations.finance_staging.dim_taxonomy_staging") \
    .select("linkrole", "gaap_version", "child_label", "parent_label") \
    .dropDuplicates()

# Initialize hierarchy
hierarchy_df = df.withColumn("level_1", F.col("parent_label")) \
    .select("linkrole", "gaap_version", "child_label", "level_1")

max_levels = 20

current_df = hierarchy_df

for i in range(2, max_levels + 1):
    parent_alias = f"p{i}"

    next_df = current_df.alias("c") \
        .join(
            df.alias(parent_alias),
            (
                (F.col(f"c.level_{i-1}") == F.col(f"{parent_alias}.child_label")) &
                (F.col("c.linkrole") == F.col(f"{parent_alias}.linkrole")) &
                (F.col("c.gaap_version") == F.col(f"{parent_alias}.gaap_version"))
            ),
            "left"
        ) \
        .withColumn(f"level_{i}", F.col(f"{parent_alias}.parent_label")) \
        .select("c.*", f"level_{i}")

    current_df = next_df

In [0]:
# Get all level columns
level_cols = [c for c in current_df.columns if c.startswith("level_")]

# Sort them numerically
level_cols_sorted = sorted(level_cols, key=lambda x: int(x.split("_")[1]))

# Reverse order (so highest level comes first)
reversed_levels = level_cols_sorted[::-1]

# Build new column expressions
new_cols = [
    F.col("linkrole"),
    F.col("gaap_version"),
    F.col("child_label")
]

for i, col_name in enumerate(reversed_levels, start=1):
    new_cols.append(F.col(col_name).alias(f"level_{i}"))

# Select reordered columns
final_df = current_df.select(*new_cols)

In [0]:
num_levels = len(reversed_levels)

# Create array of levels
# final_df = final_df.withColumn(
#     "levels_array",
#     F.array(*[F.col(f"level_{i}") for i in range(1, num_levels + 1)])
# )

final_df = final_df.withColumn(
    "levels_array",
    F.array(
        *[F.col(f"level_{i}") for i in range(1, num_levels + 1)],
        F.col("child_label")   # 👈 append leaf node
    )
)



# Remove nulls
final_df = final_df.withColumn(
    "levels_array",
    F.expr("filter(levels_array, x -> x is not null)")
)

# Re-expand safely using get()
for i in range(num_levels):
    final_df = final_df.withColumn(
        f"level_{i+1}",
        F.expr(f"get(levels_array, {i})")
    )

# Drop helper column
final_df = final_df.drop("levels_array")

In [0]:
final_df.createOrReplaceTempView('df')

In [0]:
final_df = spark.sql(f"""
select
     bigint(substr(xxhash64(concat_ws('|', child_label)), 1, 18))        AS terse_label_bigint_key
    ,sha2(concat_ws('|', child_label), 256)                               AS terse_label_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', gaap_version)), 1, 18))      AS gaap_version_bigint_key
    ,sha2(concat_ws('|', gaap_version), 256)                            AS gaap_version_key_hash
    ,bigint(substr(xxhash64(concat_ws('|', linkrole)), 1, 18))          AS linkrole_bigint_key
    ,sha2(concat_ws('|', linkrole), 256)                                AS linkrole_key_hash
    ,gaap_version
    ,linkrole
    ,child_label                                                          AS terse_label
    ,level_1                                                             AS report_label
    ,level_2                                                             AS terse_label_level_1
    ,level_3                                                             AS terse_label_level_2
    ,level_4                                                             AS terse_label_level_3
    ,level_5                                                             AS terse_label_level_4
    ,level_6                                                             AS terse_label_level_5
    ,level_7                                                             AS terse_label_level_6
    ,level_8                                                             AS terse_label_level_7
    ,level_9                                                             AS terse_label_level_8
    ,level_10                                                            AS terse_label_level_9
    ,level_11                                                            AS terse_label_level_10
    ,level_12                                                            AS terse_label_level_11
    ,level_13                                                            AS terse_label_level_12
    ,level_14                                                            AS terse_label_level_13
    ,level_15                                                            AS terse_label_level_14
    ,level_16                                                            AS terse_label_level_15
    ,level_17                                                            AS terse_label_level_16
    ,level_18                                                            AS terse_label_level_17
    ,level_19                                                            AS terse_label_level_18
    ,level_20                                                            AS terse_label_level_19
from df
""")

In [0]:
final_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{target_catalog}.{schema}.{table_name}")